## TP 2 — Profiling & périmètre · lun 25/08 (A. Abarji, journée)
> Objectif : connaître le jeu de données à fond et décider sur quoi porte NutriScope.
1. Profiling systématique des données récupérées au TP 1 : distributions, cardinalités, doublons de codes-barres,
incohérences d'unités, valeurs impossibles (sucres > 100 g/100 g, énergies nulles…).
2. Inventaire des colonnes : lesquelles servent le produit (score, substitution, images, assistant), lesquelles sont du
bruit. S'appuyer sur data-fields.txt d'Open Food Facts.
3. Décision de périmètre en équipe : rayons couverts au lancement (5 à 8 catégories), colonnes conservées, seuil de
complétude minimal par produit.
4. Rédaction de docs/perimetre.md : périmètre retenu, critères, et surtout ce qu'on écarte et pourquoi.
5. Tour des équipes en fin de journée : chaque périmètre est challengé par une autre équipe.
**À committer** : notebook de profiling + docs/perimetre.md argumenté.
> Un périmètre trop large en août se paie en janvier. Le formateur joue le client : il pousse à couper.


##### Récupération des types de données

In [1]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import plotly.express as px

file = "../data/food.parquet"

In [ ]:
# DuckDB : liste des colonnes et de leur type exact
description = duckdb.sql(f"DESCRIBE SELECT * FROM '{file}';").df()
description.to_csv("../data/data_types.csv", index=False, encoding="utf-8")

# Pour une colonne imbriquée en particulier
column_description = duckdb.sql(f"SELECT typeof(nutriments), typeof(packaging) FROM '{file}' LIMIT 1;").df()
for i in range(1) :
    print(column_description.iloc[0,i] + "\n")

### Selection des colonnes

> Voir le fichier [perimetre.md](../docs/perimetre.md)

In [ ]:
columns = ", ".join([
    # Général
    "code",
    "product_name",
    "quantity",
    "nutrition_data_per",

    # Classification
    "brands_tags",
    "categories_tags",
    "labels_tags",
    "origins_tags",

    # Ingrédients
    "ingredients_tags",
    "additives_tags",

    "nutriments",
    "nutriscore_grade",
    "nutriscore_score",
    "nutrient_levels_tags",

    "nova_group",

    # Qualité des données
    "completeness",

    # Environnement
    "environmental_score_grade",
    "environmental_score_score",

    # Images
    "images",
])

In [ ]:
output = "../data/food_france.parquet"

duckdb.sql(f"""
    COPY (
        SELECT {columns}
        FROM '{file}'
        WHERE list_contains(countries_tags, 'en:france')
    ) TO '{output}' (FORMAT PARQUET)
""")

In [5]:
df = pd.read_parquet(output, engine="pyarrow")

NameError: name 'output' is not defined

In [ ]:
print(df.shape)

#### Distributions

In [3]:
def count_for_graph(df, param):
    return df.groupby(param)[param].count().reset_index(name="somme").sort_values(param)

In [4]:
nutriscore_score = count_for_graph(df,"nutriscore_score")

graph = px.bar(nutriscore_score, x="nutriscore_score", y="somme", color="somme", color_continuous_scale="bluered")
graph.show()

NameError: name 'df' is not defined

In [20]:
nova_group = count_for_graph(df,"nova_group")

graph = px.bar(nova_group, x="nova_group", y="somme", color="somme", color_continuous_scale="bluered")
graph.show()

In [21]:
environmental_score_score = count_for_graph(df,"environmental_score_score")

graph = px.line(environmental_score_score, x="environmental_score_score", y="somme", title="Nombre de produits par score environnemental")
graph.show()

In [22]:
completeness = count_for_graph(df,"completeness")

graph = px.histogram(completeness, x="completeness", y="somme", nbins=10)
graph.show()